# Preprocessing Pipeline

## Loading the data

Code from build.py

In [ ]:
"""

When the real data is available, can run this in the terminal to get the files

uv run build.py --data-root /mnt/FastData/EXPERIMENT_DATA --out out/
uv run build.py --physics        # adds the physics-state table (~17M rows, 266 MB)
uv run build.py --adhoc-session  # a run that is not in sessions.toml (see Curation)
uv run build.py --export-eeg     # also write each aligned hyperscanner file as .fif
"""

Loading the packages and the raw data

In [ ]:
import mne
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Just copy-pasted code from my previous notebook, will need to be adapted to the new data structure

# Loading data

data_folder = Path("/work/EEG_lab/own_experiments_data")

data_raw_file_other = data_folder / 'group5_other.vhdr'

data_raw_file_own = data_folder / 'group5_own.vhdr'

# Reading sample data
raw_1 = mne.io.read_raw_brainvision(data_raw_file_other, eog=('HEOG', 'VEOG'), misc=["41"])
raw_2 = mne.io.read_raw_brainvision(data_raw_file_own, eog=('HEOG', 'VEOG'), misc=["41"])

# Define montage
montage = mne.channels.make_standard_montage('standard_1020')

# Set montage
raw_1.set_montage(montage)
raw_2.set_montage(montage)

# Load data
raw_1.load_data()
raw_2.load_data()

# Scale
scaling = dict(eeg=20e-6)

# Plot

raw_1.plot(n_channels=30, scalings=dict(eeg=50e-6))
raw_2.plot(n_channels=30, scalings=dict(eeg=50e-6))


## ICA on raw data

Parameters

In [ ]:
HP = 1;         # cut-off frequency high-pass filter [Hz] only for ICA 
LP = 40;        # cut-off frequency low-pass filter [Hz] only for ICA 
SRATE = 250;    # downsample data for ICA 
HP_ord = 500;   # high-pass filter order depends on sampling rate
LP_ord = 100;   # low-pass filter order depends on sampling rate
PRUNE = 3;      # artifact rejection threshold in SD (for ICA only)
PCA = 1;        # choose PCA option for ICA
PCADIMS = 50;   # PCA dimension if PCA option is true

**Looping over subjects:**

Filtering the data for ICA training

Epoching data and removing artificial epochs for ICA

Running ICA with the runica algorithm and the additional option of PCA

Saving ICA weights to original data


In [ ]:
# Just copy-pasted code from the matlab code 
# (from the paper "Source-Modeling Auditory Processes of EEG Data Using EEGLAB and Brainstorm"),
#  will need to be adapted to python

for s=1:len 
    # define subject name (based on set-files in PATHIN)
    subj{s} = strrep(list(s).name, '.set', '');
    
    # load rawdata (already saved as set file)
    EEG = pop_loadset('filename', [subj{s}, '.set'], 'filepath', PATHIN);
    EEG.setname = [subj{s}, '_dummy_ICA']; 

    # apply low pass filter
    EEG = pop_firws(EEG, 'fcutoff', LP, 'ftype', 'lowpass', 'wtype', 'hann' , 'forder', LP_ord);
    # downsample data - may be an optional step
    EEG = pop_resample(EEG, SRATE);
    # apply high pass filter
    EEG = pop_firws(EEG, 'fcutoff', HP, 'ftype', 'highpass', 'wtype', 'hann',...
        'forder', HP_ord);
    
    # create dummy events and epoch data to these dummy events
    EEG = eeg_regepochs(EEG, 'recurrence', 1, 'eventtype', '999');
    EEG = eeg_checkset(EEG, 'eventconsistency');
    
    # remove epochs with artefacts to improve ICA training
    EEG = pop_jointprob(EEG, 1, [1:size(EEG.data,1)], PRUNE, PRUNE, 0, 1, 0);

    # run ICA optional with our without PCA
    # a window will pop-up as soon as ICA starts which allows to interrupt
    # the ICA process. Please only press if you want to cancel the process
    if PCA == 1
        EEG = pop_runica(EEG, 'icatype', 'runica', 'extended', 1, 'pca',...
            PCADIMS);
    else
        EEG = pop_runica(EEG, 'icatype', 'runica', 'extended', 1);
    end
    
    # store ICA weights in temporary variables
    icawinv = EEG.icawinv;
    icas = EEG.icasphere;
    icaw = EEG.icaweights;
    
    # load original rawdata
    EEG = pop_loadset('filename', [subj{s}, '.set'], 'filepath', PATHIN);
      
    # write ICA weights to rawdata
    EEG.icawinv = icawinv;
    EEG.icasphere = icas;
    EEG.icaweights = icaw;
    EEG = eeg_checkset(EEG);
    
    # save new dataset with ICA weights
    EEG.setname = [subj{s}, '_ica'];
    EEG = pop_saveset(EEG, [EEG.setname, '.set'], PATHOUT);  
end

## Applying the ICA components to the data using CORRMAP 

Saving all ICA components for each subject as a PNG graphic for inspection of ICA components. Selecting a template component manually for each artifact from these figures

In [ ]:
for s = 1:len # for each subject
    # load dataset 
    subj{s}= strrep(list(s).name, '.set', '');
    EEG = pop_loadset('filename', [subj{s}, '.set'], 'filepath', PATHIN);
    
    # plot ICA components for each subject and save figure as PNG graphic for 
    # later inspections
    pop_topoplot(EEG, 0, [1:size(EEG.icawinv,2)], [subj{s}], 0, 'electrodes',...
        'off');
    filename = strrep(subj{s}, '_ica_comps', '');
    saveas(gcf,[PATHIN, filename], 'png');
    close;
end

Creating study with all datasets containing ICA weights

In [ ]:
# STUDY parameters
studyname = 'ICA_Comps';
# initialize STUDY index
index = 1; 
STUDY = []; CURRENTSTUDY = 0; ALLEEG=[]; EEG=[]; CURRENTSET=[];
 
# set memory options to allow to open more than one dataset: 
# in pop_editoptions set 'option_storedisk', 1 or use eeglab GUI
for s = 1:len # for each subject
    # load dataset in study
    subj{s}= strrep(list(s).name, '.set', '');
    dataset = [PATHIN, subj{s}, '.set'];
    [STUDY ALLEEG] = std_editset(STUDY, ALLEEG, 'name', studyname,...
        'commands',{{'index' index 'load' dataset 'subject' subj{s}}},...
        'updatedat', 'off', 'savedat', 'off', 'filename', [PATHIN, studyname]);
    index = index + 1;
    
end
 
CURRENTSTUDY = 1; EEG = ALLEEG; CURRENTSET = [1:length(EEG)];
STUDY.design = [];
[STUDY, ALLEEG] = std_checkset(STUDY, ALLEEG);
 
# optional: save study
# [STUDY EEG] = pop_savestudy( STUDY, EEG, 'filename',[studyname, '.study'],...
# 'filepath',PATHIN);

Running CORRMAP to find ICA components based on templates for each subject: EMG, eye blink, lateral eye movements

In [ ]:
# defintion of template datasets and components for each of the three
# components that should be used by the semi-automatic corrmap algorithm
blink_set = 8;    # identified set for the template eye blink component
blink_comp = 1;   # identified template component included in eye_set
heart_set = 8;    # identified set for the template heart beat  component
heart_comp = 10;  # identified template component included in heart_set
eyemov_set = 8;   # identified set for the template eye movement component
eyemov_comp = 17; # identified template component included in eye_set

In [ ]:
# use corrmap to find components in each dataset (data is accessed through study)
# 1. eye blinks
[CORRMAP, STUDY, ALLEEG] = pop_corrmap(STUDY, ALLEEG, blink_set, blink_comp,...
    'th', '0.8', 'ics', 3, 'pl', '2nd', 'title', 'plot', 'clname', '', ...
    'badcomps', 'no', 'resetclusters', 'off');
eyeblink = [CORRMAP.output.sets{2} CORRMAP.output.ics{2}];
Corr_eye = CORRMAP;
 
# 2. heartbeats (EMG in our case)
[CORRMAP, STUDY, ALLEEG] = pop_corrmap(STUDY, ALLEEG, heart_set, heart_comp,...
    'th', '0.8', 'ics', 3, 'pl', '2nd', 'title', 'plot', 'clname', '', ...
    'badcomps', 'no', 'resetclusters', 'off');
heartbeat = [CORRMAP.output.sets{2} CORRMAP.output.ics{2}];
Corr_heart = CORRMAP;
 
# 3. lateral eye movements
[CORRMAP, STUDY, ALLEEG] = pop_corrmap(STUDY,ALLEEG, eyemov_set, eyemov_comp,...
    'th', '0.8', 'ics', 3, 'pl', '2nd', 'title', 'plot', 'clname', '', ...
    'badcomps', 'no', 'resetclusters', 'off');
eyemovement = [CORRMAP.output.sets{2} CORRMAP.output.ics{2}];
Corr_lateye = CORRMAP;
 
# save info about components (optional) as MAT-file
save('components.mat', 'eyeblink', 'heartbeat' , 'eyemovement');
save('corrmap_info.mat', 'Corr_eye',  'Corr_heart', 'Corr_lateye');

Reloading data file with saved ICA weights:
    storing selected components as EEG.badcomps in each dataset
    removing selected components
    saving new dataset in PATHOUT with rejected components

In [ ]:
# find components marked for each subject, store components in EEG.badcomps and remove selected components from dataset
for s = 1:len
    # load dataset with ICA weights again
    subj{s}= strrep(list(s).name, '.set', '');
    EEG = pop_loadset('filename', [subj{s}, '.set'], 'filepath', PATHIN);
    
    # find selected components for each subject and save as badcomps
    eye = eyeblink(find(eyeblink(:,1) == s), 2)';
    heart = heartbeat(find(heartbeat(:,1) == s), 2)';
    eyemov = eyemovement(find(eyemovement(:,1) == s), 2)';
    
    # store all components in EEG structure
    EEG.badcomps = [eye heart eyemov];
    
    # plot components again, this time with note about badcomps and save as
    # PNG in PATHOUT folder
    pop_topoplot(EEG, 0, [1:size(EEG.icawinv,2)], [subj{s}, ...
        ' removed components: ', num2str(EEG.badcomps)], [6 10], 0, ...
        'electrodes', 'off');
    filename = strrep(subj{s}, '_ica', '_badcomps');
    saveas(gcf, [PATHOUT, filename], 'png');
    close
    
    # remove selected components from dataset
    EEG = pop_subcomp(EEG, [EEG.badcomps], 0);
    EEG = eeg_checkset(EEG);
    
    # save ICA cleaned dataset in ana02
    EEG.setname = [subj{s}, '_cleaned'];
    EEG = pop_saveset( EEG, 'filename', [EEG.setname, '.set'], 'filepath',...
        PATHOUT);
end
cd(fullfile(MAINPATH, 'scripts', filesep));  

## Preprocessing

Parameters

In [ ]:
PRUNE = 4;                  # artifact rejection threshold in SD
EPOCH_ON = -0.2;            # epoch start
EPOCH_OFF =0.8;             # epoch end
HP_erp = 0.1;               # cut-off frequency high-pass filter [Hz]
LP_erp = 40;                # cut-off frequency low-pass filter [Hz]
HP_ord = 500;               # high-pass filter order depends on sampling rate
LP_ord = 100;               # low-pass filter order depends on sampling rate
ev = 'S 10';                # event marker for epoching
fs = 1000;                  # sampling rate of EEG recording
baseline = [EPOCH_ON*fs 0]; # definition of baseline

**Preprocessing data after ICA cleaning:**

Filtering the data

Epoching

Rejecting contaminated epochs

In [ ]:
for s = 1:len
    # load ICA cleaned dataset
    subj{s} = strrep(list(s).name, '_ica_cleaned.set', '');
    EEG = pop_loadset('filename', [subj{s}, '_ica_cleaned.set'], 'filepath',...
        PATHIN);
    
    # apply low and high pass filter
    EEG = pop_firws(EEG, 'fcutoff', LP_erp, 'ftype', 'lowpass', 'wtype', ...
        'hann', 'forder', LP_ord);
    EEG = pop_firws(EEG, 'fcutoff', HP_erp, 'ftype', 'highpass', 'wtype', ...
        'hann', 'forder', HP_ord);
    
    # optional step: remove unnecessary event marker (e.g. here fixation cross)
    # find all events (but S 10 which is the stimulus onset) and save
    # position of EEG.event
    c=1;
    idx = [];
    for e = 1 : length(EEG.event)
        if ~ strcmp(EEG.event(e).type,ev)
            idx(c) = e;
            c=c+1;
        end
    end
    # remove all events which are not the stimulus onset (here event marker 10)
    EEG = pop_editeventvals(EEG,'delete',idx);
    EEG = eeg_checkset(EEG);
    
    # epoching and baseline
    EEG = pop_epoch(EEG, {ev}, [EPOCH_ON EPOCH_OFF], 'newname', EEG.setname,...
        'epochinfo', 'yes');
    EEG = pop_rmbase(EEG, baseline);
    
    # artifact rejection based on joint probability
    # reject artifacts with joint prob. > PRUNE (SD)
    # optional: store the indices of epoch that are going to be rejected
    # otherwise reject epochs immediately:
    # EEG = pop_jointprob(EEG, 1, [1:EEG.nbchan], PRUNE, PRUNE, 0, 1 , 0);
    EEG = pop_jointprob(EEG, 1, [1:EEG.nbchan], PRUNE, PRUNE, 0, 0 , 0);
    EEG = eeg_rejsuperpose( EEG, 1, 1, 1, 1, 1, 1, 1, 1);
    rej_ep(s) = {find(EEG.reject.rejglobal == 1)};
    EEG = pop_rejepoch( EEG, EEG.reject.rejglobal ,0);
    
    # optional: get indices of ICA components that were rejected:
    comps(s) = {EEG.badcomps};
    
    # save pruned and epoched data set
    EEG = eeg_checkset(EEG);
    EEG.setname = [subj{s}, '_ep_ar'];
    EEG = pop_saveset(EEG, [EEG.setname, '.set'], PATHOUT);
    
end
# optional: save rejected epochs per subject and ICA components that were rejected
save([PATHOUT 'info'], 'rej_ep','comps' )